## **Speed Dating - Data Storytelling**

Source unique: `outputs/data/df_speed_dating_cleaned.csv`

Ce notebook rassemble les indicateurs, graphiques et messages clés dans un enchaînement clair: situation de départ, points de friction, leviers de conversion, segments à activer et conclusion actionnable.


### **Fil de lecture**

1. Situation de départ (chiffres clés)
2. Friction entre intention et conversion
3. Leviers les plus utiles pour agir
4. Cohérence entre déclaratif et comportement
5. Segments et ajustements de communication
6. Conclusion opérationnelle

> Convention d’export: chaque visuel enregistré suit le format `viz_<theme>_<message>.png` dans `outputs/images`.


In [35]:
# ============================================================================
# SETUP
# ============================================================================
from pathlib import Path
import shutil
import subprocess
import tempfile

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path("../")
INPUT_PATH = PROJECT_ROOT / "outputs" / "data" / "df_speed_dating_cleaned.csv"
OUT_IMG = PROJECT_ROOT / "outputs" / "images"
OUT_IMG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)

if "gender_label" not in df.columns and "gender" in df.columns:
    df["gender_label"] = df["gender"].map({0: "Femme", 1: "Homme"})
if "match_label" not in df.columns and "match" in df.columns:
    df["match_label"] = df["match"].map({0: "Pas de match", 1: "Match"})
if "gender" in df.columns:
    df["segment_genre"] = df["gender"].map({1: "Homme -> Femme", 0: "Femme -> Homme"})

ATTR_MAP = {
    "attr": "Attractivite", "sinc": "Sincerite", "intel": "Intelligence",
    "fun": "Fun", "amb": "Ambition", "shar": "Interets communs"
}


def export_figure(fig, name_no_ext: str, width: int = 4200, height: int = 2400, scale: int = 2):
    """Export PNG: Plotly/Kaleido puis fallback Chromium headless."""
    png_path = OUT_IMG / f"{name_no_ext}.png"

    try:
        fig.write_image(str(png_path), width=width, height=height, scale=scale)
        print(f"Export image (kaleido): {png_path}")
        return
    except Exception as kaleido_err:
        print(f"Kaleido indisponible ({type(kaleido_err).__name__}), fallback Chromium...")

    chromium_bin = shutil.which("chromium") or shutil.which("chromium-browser") or shutil.which("google-chrome")
    if not chromium_bin:
        raise RuntimeError("Export PNG impossible: ni Kaleido fonctionnel ni Chromium detecte.") from kaleido_err

    with tempfile.NamedTemporaryFile(suffix=".html", delete=False) as tmp:
        tmp_html = Path(tmp.name)

    try:
        fig.write_html(str(tmp_html), include_plotlyjs="cdn")
        cmd = [
            chromium_bin, "--headless", "--disable-gpu", "--no-sandbox",
            f"--window-size={width},{height}", f"--screenshot={png_path}", tmp_html.resolve().as_uri()
        ]
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"Export image (chromium): {png_path}")
    finally:
        try:
            tmp_html.unlink(missing_ok=True)
        except Exception:
            pass

print(f"Dataset: {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"Source: {INPUT_PATH}")

Dataset: 8,283 lignes x 50 colonnes
Source: ../outputs/data/df_speed_dating_cleaned.csv


### **1. Situation de départ**

Point de repère: taille du jeu de données, niveau moyen de match et écart entre intention de poursuivre et conversion effective.


In [36]:
# Stats descriptives
print("\n" + "─" * 65)
print("Statistiques descriptives")
print("─" * 65)
 
summary_cols = [c for c in ["match", "dec", "attr", "fun", "like", "prob", "int_corr", "samerace", "age", "go_out"] if c in df.columns]
print(df[summary_cols].describe(include="all").T)

match_rate = float(df["match"].mean() * 100)
dec_rate = float(df["dec"].mean() * 100) if "dec" in df.columns else np.nan
print(f"\nTaux de match global: {match_rate:.1f}%")
print(f"Taux de 'Oui' (dec=1): {dec_rate:.1f}%")

if "gender_label" in df.columns:
    print("\nTaux de match par genre:")
    print((df.groupby("gender_label")["match"].mean() * 100).round(2))


─────────────────────────────────────────────────────────────────
Statistiques descriptives
─────────────────────────────────────────────────────────────────
           count       mean       std    min    25%    50%    75%    max
match     8283.0   0.164433  0.370691   0.00   0.00   0.00   0.00   1.00
dec       8283.0   0.421224  0.493785   0.00   0.00   0.00   1.00   1.00
attr      8283.0   6.193094  1.931032   0.00   5.00   6.00   8.00  10.00
fun       8283.0   6.412411  1.913320   0.00   5.00   7.00   8.00  10.00
like      8283.0   6.130774  1.816015   0.00   5.00   6.00   7.00  10.00
prob      8283.0   5.207835  2.093503   0.00   4.00   5.00   7.00  10.00
int_corr  8283.0   0.195180  0.302263  -0.83  -0.01   0.21   0.43   0.91
samerace  8283.0   0.398165  0.489549   0.00   0.00   0.00   1.00   1.00
age       8283.0  26.358928  3.566763  18.00  24.00  26.00  28.00  55.00
go_out    8283.0   2.157189  1.105918   1.00   1.00   2.00   3.00   7.00

Taux de match global: 16.4%
Taux de '

### **2. Lecture des leviers de conversion**

Chaque bloc suit la même logique: contexte, visuel, observation, interprétation et décision à envisager.


In [37]:
# Convention d'export pour la presentation
def export_viz(fig, suffix, width=1200, height=700, scale=2):
    export_figure(fig, f"viz_{suffix}", width=width, height=height, scale=scale)

print("Convention export activee: viz_<theme>_<message>.png")


Convention export activee: viz_<theme>_<message>.png


### **2.1 Intention vs conversion réelle**

Comparer le volume de « Oui » (`dec`) avec le taux de match montre à quel niveau la conversion se perd.


In [38]:
dec_rate = float(df["dec"].mean() * 100) if "dec" in df.columns else np.nan
match_rate = float(df["match"].mean() * 100)

funnel_df = pd.DataFrame({
    "Etape": ["Intention: dec=Oui", "Conversion: match"],
    "Taux (%)": [dec_rate, match_rate],
})

fig_viz_funnel = px.bar(
    funnel_df,
    x="Etape", y="Taux (%)", text="Taux (%)", color="Etape",
    title="Du Oui au match: où se perd la conversion",
    color_discrete_sequence=["#378ADD", "#1D9E75"],
)
fig_viz_funnel.update_layout(showlegend=False, height=750)
fig_viz_funnel.show()
export_viz(fig_viz_funnel, "conversion_oui_vers_match")

Export image (kaleido): ../outputs/images/viz_conversion_oui_vers_match.png


**Observation et interprétation**

- Le taux de `dec=Oui` est supérieur au taux de `match`.
- Le principal enjeu est donc l’alignement mutuel, pas uniquement l’intention individuelle.
- **Priorité**: travailler l’ajustement entre profils proposés, promesse perçue et moment de relance.


### **2.2 Signal prioritaire pour piloter les actions**

Le score `like` sert de signal simple pour hiérarchiser relances et priorités.


In [39]:
thresholds = [4, 5, 6, 7, 8, 9]
rates = [df[df["like"] >= t]["match"].mean() * 100 for t in thresholds]
like_curve = pd.DataFrame({"Seuil like": thresholds, "Taux match (%)": rates})

fig_viz_like = px.line(
    like_curve,
    x="Seuil like", y="Taux match (%)", markers=True,
    title="Effet du score like sur la conversion",
)
fig_viz_like.update_layout(height=750)
fig_viz_like.show()
export_viz(fig_viz_like, "levier_like_conversion")

Export image (kaleido): ../outputs/images/viz_levier_like_conversion.png


**Observation et interprétation**

- La probabilité de match monte avec le score `like`.
- Ce signal est opérationnel: il transforme une intuition en règle d’action.
- **Priorité**: bâtir des relances graduées selon le niveau de `like`.


### **2.3 Déclaratif et comportement observé**

Comparer ce qui est annoncé avant la rencontre et ce qui est noté pendant l’échange aide à ajuster les messages.


In [40]:
pairs = [
    ("attr1_1", "attr", "Attractivite"),
    ("sinc1_1", "sinc", "Sincerite"),
    ("intel1_1", "intel", "Intelligence"),
    ("fun1_1", "fun", "Fun"),
    ("amb1_1", "amb", "Ambition"),
    ("shar1_1", "shar", "Interets communs"),
]
rows = []
for dcol, rcol, label in pairs:
    if {dcol, rcol}.issubset(df.columns):
        rows.append({
            "Variable": label,
            "Importance declarée": float(df[dcol].mean()),
            "Importance réelle": float(df[rcol].mean()),
        })
cmp = pd.DataFrame(rows)
cmp_m = cmp.melt(id_vars="Variable", var_name="Type", value_name="Score moyen")

fig_viz_decl_real = px.bar(
    cmp_m,
    x="Variable", y="Score moyen", color="Type", barmode="group",
    title="Déclaratif vs vécu: ce qui pèse dans la conversion",
    color_discrete_map={"Importance declarée": "#7F77DD", "Importance réelle": "#EF9F27"},
)
fig_viz_decl_real.update_layout(height=780)
fig_viz_decl_real.show()
export_viz(fig_viz_decl_real, "declaratif_vs_vecu")

Export image (kaleido): ../outputs/images/viz_declaratif_vs_vecu.png


**Observation et interprétation**

- Les préférences déclarées avant rencontre ne reflètent pas toujours les notes observées ensuite.
- Pour décider, les signaux comportementaux sont plus fiables que le déclaratif seul.
- **Priorité**: adapter wording, onboarding et relance aux comportements réels.


### **2.4 Priorités par segment**

Lecture complémentaire: les priorités déclarées ne se distribuent pas exactement de la même façon selon le genre.


In [41]:
## Priorités déclarées par genre
pairs = [
    ("attr1_1", "attr", "Attractivite"),
    ("sinc1_1", "sinc", "Sincerite"),
    ("intel1_1", "intel", "Intelligence"),
    ("fun1_1", "fun", "Fun"),
    ("amb1_1", "amb", "Ambition"),
    ("shar1_1", "shar", "Interets communs"),
]

pref_cols = [p for p, _, _ in pairs if p in df.columns]
pref_gender = df.groupby("gender_label", dropna=False)[pref_cols].mean().T.reset_index().rename(columns={"index": "pref_col"})
pref_gender["Attribut"] = pref_gender["pref_col"].map(dict((p, l) for p, _, l in pairs))
long_q1 = pref_gender.melt(id_vars=["pref_col", "Attribut"], var_name="Genre", value_name="Importance")

fig_q1 = px.bar(
    long_q1,
    x="Attribut", y="Importance", color="Genre", barmode="group",
    title="Priorités déclarées selon le genre",
    color_discrete_map={"Femme": "#D4537E", "Homme": "#378ADD"}
)
fig_q1.update_layout(height=700)
fig_q1.show()
export_viz(fig_q1, "priorites_genre")

print("Lecture:")
print("- Les barres les plus basses representent les attributs les moins prioritaires dans le declaratif.")
print("- Les differences Femme/Homme indiquent si le 'least desirable' change selon le genre.")

Export image (kaleido): ../outputs/images/viz_priorites_genre.png
Lecture:
- Les barres les plus basses representent les attributs les moins prioritaires dans le declaratif.
- Les differences Femme/Homme indiquent si le 'least desirable' change selon le genre.


### **2.5 Repères contextuels**

Les corrélations contextuelles permettent de hiérarchiser les signaux à conserver en suivi régulier.


In [42]:
## Poids relatif des signaux contextuels
q3 = pd.DataFrame({
    "Variable": ["Interets communs (int_corr)", "Meme race (samerace)"],
    "Correlation avec match": [
        float(df["int_corr"].corr(df["match"])) if {"int_corr", "match"}.issubset(df.columns) else np.nan,
        float(df["samerace"].corr(df["match"])) if {"samerace", "match"}.issubset(df.columns) else np.nan,
    ]
})
fig_q3 = px.bar(q3, x="Variable", y="Correlation avec match", color="Variable", title="Intérêts en commun vs même origine: poids relatif")
fig_q3.update_layout(showlegend=False, height=700)
fig_q3.show()
export_viz(fig_q3, "interets_vs_origine")

print("Lecture:")
print("- La variable avec correlation absolue la plus forte est la plus informative pour predire un second date.")

Export image (kaleido): ../outputs/images/viz_interets_vs_origine.png
Lecture:
- La variable avec correlation absolue la plus forte est la plus informative pour predire un second date.


In [43]:
## Confiance perçue et résultat réel
if {"prob", "match"}.issubset(df.columns):
    calib = df[["prob", "match"]].dropna().copy()
    calib["prob_bin"] = pd.cut(calib["prob"], bins=[0,2,4,6,8,10], include_lowest=True)
    calib_grp = calib.groupby("prob_bin", observed=False).agg(
        prob_mean=("prob", "mean"),
        match_rate=("match", "mean"),
        n=("match", "size"),
    ).reset_index()
    calib_grp["match_rate_pct"] = (calib_grp["match_rate"] * 100).round(1)

    fig_q4 = px.line(
        calib_grp,
        x="prob_mean", y="match_rate_pct", markers=True,
        title="Confiance perçue et résultat réel",
        labels={"prob_mean": "Probabilite percue moyenne", "match_rate_pct": "Taux de match reel (%)"}
    )
    fig_q4.add_trace(go.Scatter(x=[0,10], y=[0,100], mode="lines", name="Calibration parfaite", line=dict(dash="dash", color="#888780")))
    fig_q4.update_layout(height=700)
    fig_q4.show()
    export_viz(fig_q4, "confiance_vs_resultat")

    print("Lecture:")
    print("- Plus la courbe est proche de la diagonale, meilleure est l'auto-evaluation du marche relationnel.")

Export image (kaleido): ../outputs/images/viz_confiance_vs_resultat.png
Lecture:
- Plus la courbe est proche de la diagonale, meilleure est l'auto-evaluation du marche relationnel.


### **2.6 Segments à activer**

Les écarts de conversion selon le sens de la rencontre permettent de prioriser des messages plus ciblés.


In [44]:
if "segment_genre" not in df.columns and "gender" in df.columns:
    df["segment_genre"] = df["gender"].map({1: "Homme -> Femme", 0: "Femme -> Homme"})

seg = (df.groupby("segment_genre")["match"].mean() * 100).reset_index(name="Taux de match (%)")
fig_viz_seg = px.bar(
    seg,
    x="segment_genre", y="Taux de match (%)", color="segment_genre",
    title="Segmentation: niveaux de conversion",
    color_discrete_map={"Homme -> Femme": "#378ADD", "Femme -> Homme": "#D4537E"},
)
fig_viz_seg.update_layout(showlegend=False, height=760)
fig_viz_seg.show()
export_viz(fig_viz_seg, "segmentation_conversion")

Export image (kaleido): ../outputs/images/viz_segmentation_conversion.png


**Observation et interprétation**

- Des écarts de conversion existent entre segments.
- Une approche unique perd en efficacité face à des attentes différentes.
- **Priorité**: personnaliser messages, timing et créatifs par segment.


In [45]:
# Disponibilité des variables d’ordre de passage
order_cols = [c for c in df.columns if any(k in c.lower() for k in ["order", "position", "first", "last", "date_no", "round"])]
print("Variables d’ordre détectées:", order_cols)

if len(order_cols) == 0:
    print("Aucun test possible ici: pas de variable d’ordre exploitable dans les données nettoyées.")
else:
    print("Une variable d'ordre existe: ajouter une analyse match rate par ordre.")

Variables d’ordre détectées: []
Aucun test possible ici: pas de variable d’ordre exploitable dans les données nettoyées.


## **3. Conclusion**

### **Ce qui ressort clairement**
- La conversion se joue surtout entre l’intention individuelle (`dec`) et l’intérêt mutuel (`match`).
- Les signaux vécus pendant la rencontre pèsent davantage que les déclarations initiales.
- Le score `like` permet une priorisation simple et directement exploitable.
- La segmentation améliore la pertinence des prises de parole et des relances.

### **Décisions à privilégier**
1. Prioriser les actions de suivi sur les profils à fort potentiel (`like` élevé).
2. Construire des messages qui reflètent les signaux observés, pas seulement les intentions déclarées.
3. Adapter les scénarios de communication selon les segments.
4. Suivre régulièrement les visuels `viz_*` comme tableau de pilotage.
